In [1]:
import pandas as pd
seq_regions = ['SEQ_H1', 'SEQ_H2', 'SEQ_L1', 'SEQ_L2', 'SEQ_L3']
cf_regions = ['CF_H1', 'CF_H2', 'CF_L1', 'CF_L2', 'CF_L3']
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset=seq_regions)
    .dropna(subset=cf_regions)
    .drop_duplicates(subset=seq_regions) 
)
antigen_counts = df["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
df = df[df["antigen_name"].isin(antigen_counts[antigen_counts >= 5].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt

In [2]:
import pandas as pd
from sklearn.metrics import v_measure_score

#dataframes are created
df_true = df
df_pred = pd.read_csv("data/MMseqs2/MMseqs2_summary_cluster.tsv", sep="\t")


idx_cols  = ["pdb", "Hchain", "Lchain"]                                 #idx columns which identify the unique combinations
true_idx  = df_true.drop_duplicates(idx_cols).set_index(idx_cols)       #removes each row in which this combinations appears more than one time in both data frames
pred_idx  = df_pred.drop_duplicates(idx_cols).set_index(idx_cols)       #set index for dataframe with the three elements

#Loop
for cdr in ["H1","H2","L1","L2","L3"]:
    col = f"CF_{cdr}"                                                   
    
    t, p = true_idx[col].align(pred_idx[col], join="inner")             #returns two series for t(true) and p(predict), that share same  index, with the same amount of lines
    
    mask = t.notna() & p.notna()                                        #remove missing values, elements are pair wise deleted
    n = mask.sum()                                                      # sum of remaining pairs
    
    print(f"CDR {cdr}: V-Measure = {v_measure_score(t[mask], p[mask]):.3f}  (n = {n})")



CDR H1: V-Measure = 0.332  (n = 712)
CDR H2: V-Measure = 0.385  (n = 502)
CDR L1: V-Measure = 0.558  (n = 712)
CDR L2: V-Measure = 0.000  (n = 712)
CDR L3: V-Measure = 0.363  (n = 703)
